# TalentIQ - Preprocessing and Feature Engineering Guide

This notebook explains everything that happens in `preprocessing.py` and `feature_engineering.py`.
It is written as a guide. You will understand not just what the code does, but why each step exists.

---

## What is preprocessing?

Raw data is never perfect. It can have missing values, duplicate rows, wrong data types, and extreme outliers.
If we feed that raw data directly into a machine learning model, the model will learn the wrong things and perform poorly.

Preprocessing is the step where we clean and prepare the data so the model can learn from it properly.

In this project, preprocessing does the following in order:
1. Load the raw CSV file
2. Validate that required columns exist
3. Fill any missing values
4. Remove duplicate rows
5. Handle outliers
6. Save the cleaned data as a new file

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load raw data to demonstrate each step
df_raw = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print('Raw shape:', df_raw.shape)

---

## Step 1 - Loading the data

**What the code does in `load_data()`:**

It reads the mode from `config.yaml`. The mode can be `sample` or `full`.
- `sample` mode loads 500 rows for quick testing
- `full` mode loads all 1470 rows for the final run

After loading, it does some immediate cleanup:

**Dropping useless columns:**
Some columns in this dataset carry zero useful information.
- `EmployeeNumber` is just an ID. It has no relationship with whether someone leaves.
- `EmployeeCount` is always 1 for every single row.
- `Over18` is always Y for every single row.
- `StandardHours` is always 80 for every single row.

These columns do not help the model learn anything. Keeping them would just add noise.

**Converting the target column:**
The `Attrition` column originally has text values: Yes or No.
Machine learning models only understand numbers.
So we convert: Yes becomes 1 and No becomes 0.

**Converting ordinal columns to text:**
Some columns like `JobSatisfaction` and `Education` are stored as integers (1, 2, 3, 4).
But those numbers have a specific meaning (Low, Medium, High, Very High).
We convert them to meaningful text labels here so that the ordinal encoder can later learn the correct order.

In [ ]:
# Demonstrate what load_data() does step by step

df = df_raw.copy()

# Drop useless constant columns
drop_cols = ['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print('After dropping useless columns:', df.shape)

# Convert target to numeric
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
print('Attrition value counts after conversion:')
print(df['Attrition'].value_counts())

# Convert integer ordinal columns to string labels
col_maps = {
    'Education': {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'},
    'JobSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'WorkLifeBalance': {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'},
}
for col, mapping in col_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

print('\nJobSatisfaction sample values after mapping:')
print(df['JobSatisfaction'].value_counts())

---

## Step 2 - Validating the data

**What the code does in `validate_data()`:**

Before we process the data, we check if the data is actually usable.
This is a safety check. If someone gives us a wrong file, the pipeline should fail loudly with a clear error message rather than producing wrong results silently.

Three things are checked:
1. Is the dataframe empty?
2. Does the target column (Attrition) exist?
3. Do all required feature columns exist?

The list of required columns comes from `features.yaml`, not from the code itself.
This makes the validation flexible. If we add a new feature to the YAML, the validation automatically updates.

In [ ]:
# Show what validate_data() checks

required_numerical = [
    'Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
    'YearsWithCurrManager', 'NumCompaniesWorked', 'YearsSinceLastPromotion'
]
required_categorical = ['BusinessTravel', 'Department', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
target = 'Attrition'

all_required = required_numerical + required_categorical + [target]
missing = [col for col in all_required if col not in df.columns]

if missing:
    print('Missing columns:', missing)
else:
    print('Validation passed. All required columns are present.')
print('DataFrame is empty:', df.empty)

---

## Step 3 - Handling missing values

**What the code does in `handle_missing_values()`:**

Even if this particular dataset has no missing values right now, we write the code defensively.
Real world data often has gaps and we want this pipeline to handle any dataset.

**Strategy used:**
- For numerical columns (like Age, MonthlyIncome): fill missing values with the **median**.
  - Why median and not mean? Because the mean is pulled by outliers. If one employee earns 500,000 and the rest earn 5,000, the mean is misleading. The median gives a more accurate middle value.
  
- For categorical columns (like Department, JobRole): fill with the **mode** (most common value).
  - Why mode? Because for text categories, you cannot average. The most common value is the safest guess.

In [ ]:
# Demonstrate the logic of handle_missing_values()

# Artificially introduce missing values to show the logic
df_demo = df.copy()
df_demo.loc[0:10, 'MonthlyIncome'] = np.nan
df_demo.loc[5:15, 'Department'] = np.nan

print('Missing values before:')
print(df_demo[['MonthlyIncome', 'Department']].isnull().sum())

# Fix numerical with median
df_demo['MonthlyIncome'] = df_demo['MonthlyIncome'].fillna(df_demo['MonthlyIncome'].median())

# Fix categorical with mode
df_demo['Department'] = df_demo['Department'].fillna(df_demo['Department'].mode()[0])

print('\nMissing values after:')
print(df_demo[['MonthlyIncome', 'Department']].isnull().sum())
print('\nMedian MonthlyIncome used:', df['MonthlyIncome'].median())
print('Mode Department used:', df['Department'].mode()[0])

---

## Step 4 - Removing duplicates

**What the code does in `remove_duplicates()`:**

If the same employee row appears twice in the dataset, we are giving the model the same example twice.
This is like a student reading the same flashcard twice in a row. It does not help learning.
It can also bias the model toward rows that appear more often.

The code drops any rows where all column values are identical to another row.
It also logs how many rows were removed so we have visibility into what happened.

In [ ]:
before = len(df)
df_dedup = df.drop_duplicates()
after = len(df_dedup)
print(f'Rows before: {before}')
print(f'Rows after: {after}')
print(f'Duplicates removed: {before - after}')

---

## Step 5 - Handling outliers with IQR

**What the code does in `handle_outliers()`:**

An outlier is a value that is extremely different from all other values in that column.
For example, if most employees earn between 2,000 and 15,000 per month but one earns 100,000, that one row can distort what the model learns.

**The IQR method:**
IQR stands for Interquartile Range. It measures the spread of the middle 50% of the data.

- Q1 = value at the 25th percentile (bottom 25% of the data)
- Q3 = value at the 75th percentile (top 25% of the data)
- IQR = Q3 - Q1
- Lower bound = Q1 - 1.5 x IQR
- Upper bound = Q3 + 1.5 x IQR

Any value below the lower bound or above the upper bound is considered an outlier.

**What we do with outliers:**
We clip them. Clipping means we replace the outlier with the boundary value.
- A value of 100,000 that exceeds the upper bound of 25,000 gets replaced with 25,000.
- We do not delete the row. We just bring the extreme value back into a reasonable range.

This is done for: `MonthlyIncome`, `TotalWorkingYears`, `YearsAtCompany`, `DistanceFromHome`

In [ ]:
# Demonstrate IQR outlier handling step by step

col = 'MonthlyIncome'
data = df[col].copy()

q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f'Column: {col}')
print(f'Q1: {q1:.0f}')
print(f'Q3: {q3:.0f}')
print(f'IQR: {iqr:.0f}')
print(f'Lower bound: {lower:.0f}')
print(f'Upper bound: {upper:.0f}')
print(f'Values above upper bound: {(data > upper).sum()}')
print(f'Max before clipping: {data.max():.0f}')

clipped = np.clip(data, lower, upper)
print(f'Max after clipping: {clipped.max():.0f}')

In [ ]:
# Visual comparison before and after clipping
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(data, bins=30, color='tomato')
axes[0].set_title('MonthlyIncome - Before Clipping')
axes[0].set_xlabel('Income')

axes[1].hist(clipped, bins=30, color='steelblue')
axes[1].set_title('MonthlyIncome - After Clipping')
axes[1].set_xlabel('Income')

plt.tight_layout()
plt.show()
print('The right tail is compressed. Extreme values are brought back to the boundary.')

---

## Step 6 - Saving the processed data

**What the code does at the end of `preprocess_data()`:**

After all the cleaning steps, the processed dataframe is saved to a CSV file.
The save path comes from `config.yaml`.

- In full mode: saved to `data/processed/processed_full.csv`
- In sample mode: saved to `data/processed/processed_sample.csv`

Why save it?
- So we do not have to run preprocessing every time we train. We can load the clean file directly.
- It also creates a checkpoint. If something breaks in training, we can inspect the processed data to see if preprocessing was correct.

---

# Feature Engineering

## What is feature engineering?

Feature engineering is the process of creating new columns from the existing ones.
The idea is that the raw columns might not directly tell the model what it needs to know.
But when you combine or transform columns in a meaningful way, you give the model more informative inputs.

Think of it like this:
- Raw feature: `YearsAtCompany = 10` and `MonthlyIncome = 3000`
- Engineered feature: `IncomePerYear = 3000 / (10 + 1) = 272` (low income relative to years spent = risk factor)

The engineered feature combines two columns into one that directly expresses a meaningful relationship.

In this project, all feature engineering is done in `feature_engineering.py` inside the `engineer_features()` function.
It takes the cleaned dataframe and adds 7 new columns.

---

In [ ]:
# Load the processed data to demonstrate feature engineering
try:
    df_proc = pd.read_csv('../data/processed/processed_full.csv')
    print('Loaded processed data:', df_proc.shape)
except FileNotFoundError:
    # If not yet saved, use raw with basic mapping applied
    df_proc = df.copy()
    print('Using raw data for demonstration:', df_proc.shape)

---

## Feature 1 - IncomePerYear

**Formula:** `MonthlyIncome / (YearsAtCompany + 1)`

**Why does this matter?**
An employee who has been with the company for 10 years but still earns a low salary might feel undervalued.
They are more likely to leave than someone who earns the same salary but has only been here for 1 year.

This feature captures that ratio. A low IncomePerYear means the employee is earning little relative to how long they stayed.

We add 1 to YearsAtCompany in the denominator to avoid dividing by zero for new employees.

In [ ]:
df_proc['IncomePerYear'] = df_proc['MonthlyIncome'] / (df_proc['YearsAtCompany'] + 1)

print('IncomePerYear statistics:')
print(df_proc['IncomePerYear'].describe().round(2))

# Show average IncomePerYear for those who left vs stayed
print('\nAverage IncomePerYear by Attrition:')
print(df_proc.groupby('Attrition')['IncomePerYear'].mean().round(2))

---

## Feature 2 - SatisfactionScore

**Formula:** Average of `JobSatisfaction`, `EnvironmentSatisfaction`, `RelationshipSatisfaction`, `WorkLifeBalance`

**Why does this matter?**
Each of these four columns measures a different aspect of how satisfied an employee is.
Instead of treating them as four separate signals, we combine them into a single overall satisfaction score.

First, the text labels are mapped back to numbers (Low=1, Medium=2, High=3, Very High=4).
Then we take the average across all four columns.
A lower average means the employee is generally dissatisfied across multiple areas, which is a strong signal for attrition.

In [ ]:
sat_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4,
           'Bad': 1, 'Good': 2, 'Better': 3, 'Best': 4}

satisfaction_cols = [c for c in ['JobSatisfaction', 'EnvironmentSatisfaction',
                                  'RelationshipSatisfaction', 'WorkLifeBalance']
                     if c in df_proc.columns]

sat_df = df_proc[satisfaction_cols].replace(sat_map).apply(pd.to_numeric, errors='coerce')
df_proc['SatisfactionScore'] = sat_df.mean(axis=1)

print('SatisfactionScore statistics:')
print(df_proc['SatisfactionScore'].describe().round(2))

print('\nAverage SatisfactionScore by Attrition:')
print(df_proc.groupby('Attrition')['SatisfactionScore'].mean().round(3))

---

## Feature 3 - LoyaltyIndex

**Formula:** `YearsWithCurrManager / (TotalWorkingYears + 1)`

**Why does this matter?**
If an employee has spent a large percentage of their total career under their current manager, it suggests a strong relationship and loyalty.
A low LoyaltyIndex means they recently switched managers or have a long career history but little time with the current manager.
This can indicate instability or dissatisfaction with the management change.

In [ ]:
df_proc['LoyaltyIndex'] = df_proc['YearsWithCurrManager'] / (df_proc['TotalWorkingYears'] + 1)

print('LoyaltyIndex statistics:')
print(df_proc['LoyaltyIndex'].describe().round(3))

print('\nAverage LoyaltyIndex by Attrition:')
print(df_proc.groupby('Attrition')['LoyaltyIndex'].mean().round(3))

---

## Feature 4 - WorkloadScore

**Formula:** `TrainingTimesLastYear x JobInvolvement (as number)`

**Why does this matter?**
An employee who is highly involved in their job AND attends many training sessions is either very engaged or very overloaded.
This score tries to capture the combined intensity of their workload and involvement.
When combined with other features, it helps the model understand burnout risk.

In [ ]:
inv_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4}
if 'JobInvolvement' in df_proc.columns:
    job_inv = df_proc['JobInvolvement'].map(inv_map).fillna(2)
else:
    job_inv = 2

df_proc['WorkloadScore'] = df_proc['TrainingTimesLastYear'] * job_inv

print('WorkloadScore statistics:')
print(df_proc['WorkloadScore'].describe().round(2))

---

## Feature 5 - CareerGrowthRate

**Formula:** `(YearsAtCompany - YearsSinceLastPromotion) / (YearsAtCompany + 1)`

**Why does this matter?**
This measures how recently an employee was promoted relative to how long they have been at the company.
If someone has been at the company for 8 years and their last promotion was 7 years ago, the numerator is small (1), and the ratio is low.
That is a red flag for attrition. The employee is stagnating.

A higher ratio means they were promoted more recently relative to their tenure, suggesting active career growth.

In [ ]:
df_proc['CareerGrowthRate'] = (
    (df_proc['YearsAtCompany'] - df_proc['YearsSinceLastPromotion']) /
    (df_proc['YearsAtCompany'] + 1)
)

print('CareerGrowthRate statistics:')
print(df_proc['CareerGrowthRate'].describe().round(3))

print('\nAverage CareerGrowthRate by Attrition:')
print(df_proc.groupby('Attrition')['CareerGrowthRate'].mean().round(3))

---

## Feature 6 - OverTimeRisk

**Formula:** `1 if OverTime == Yes, else 0`

**Why does this matter?**
From EDA, we found that OverTime is the single strongest predictor of attrition.
Employees who do overtime consistently leave at a much higher rate.

The `OverTime` column already exists in the dataset as text (Yes/No).
We create a binary numeric version called `OverTimeRisk` so the model can use it directly as a number.
This is different from the one-hot encoding that happens later. This feature is intentionally kept as a single 0/1 column.

In [ ]:
if 'OverTime' in df_proc.columns:
    df_proc['OverTimeRisk'] = (df_proc['OverTime'] == 'Yes').astype(int)
else:
    df_proc['OverTimeRisk'] = 0

print('OverTimeRisk distribution:')
print(df_proc['OverTimeRisk'].value_counts())

print('\nAttrition rate by OverTimeRisk:')
print(df_proc.groupby('OverTimeRisk')['Attrition'].mean().round(3))

---

## Feature 7 - TenureStabilityIndex

**Formula:** `TotalWorkingYears / (NumCompaniesWorked + 1)`

**Why does this matter?**
This feature measures how stable someone's career history is.
An employee who has 15 total working years but has worked at 8 different companies has a low stability index.
They are a job hopper. Historically, job hoppers are more likely to leave again.

An employee with 15 years at only 2 companies has a high stability index, suggesting they stay long term.

In [ ]:
df_proc['TenureStabilityIndex'] = df_proc['TotalWorkingYears'] / (df_proc['NumCompaniesWorked'] + 1)

print('TenureStabilityIndex statistics:')
print(df_proc['TenureStabilityIndex'].describe().round(2))

print('\nAverage TenureStabilityIndex by Attrition:')
print(df_proc.groupby('Attrition')['TenureStabilityIndex'].mean().round(3))

---

## Summary of all engineered features

In [ ]:
engineered = ['IncomePerYear', 'SatisfactionScore', 'LoyaltyIndex',
              'WorkloadScore', 'CareerGrowthRate', 'OverTimeRisk', 'TenureStabilityIndex']

print('All 7 engineered features added to the dataframe:')
print(df_proc[engineered].describe().round(3))

In [ ]:
# Show correlation of each engineered feature with Attrition
corr_with_target = df_proc[engineered + ['Attrition']].corr()['Attrition'].drop('Attrition')
corr_with_target = corr_with_target.sort_values(key=abs, ascending=False)

print('Correlation of engineered features with Attrition:')
print(corr_with_target.round(3))

---

## What happens next?

After preprocessing and feature engineering, the data goes into the training pipeline.
There, the following transformations happen:

1. **Ordinal Encoding** - Columns like Education and JobSatisfaction are encoded as ordered numbers so the model understands the rank (Low < Medium < High < Very High).

2. **One-Hot Encoding** - Columns like Department and JobRole are converted into binary columns because there is no meaningful order between categories like Sales, HR, and Engineering.

3. **Standard Scaling** - Numerical columns are rescaled to have mean=0 and standard deviation=1. This is needed so that a feature like MonthlyIncome (values in thousands) does not dominate over Age (values in tens) just because of scale.

4. **SMOTE** - Synthetic samples of the minority class (employees who left) are generated to balance the training data before the model trains.

All of this happens inside `train.py` using a `ColumnTransformer` and `imblearn`.
See the Training Pipeline notebook for the full explanation.